In [ ]:
!pip install -q transformers accelerate torch pandas

import json
import time
from datetime import datetime
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()

print(f"Model hazır: {model_id} | Cihaz: {model.device}")

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Model hazır: Qwen/Qwen2.5-3B-Instruct | Cihaz: cuda:0


In [ ]:
# 10 Soruluk Altın Standart Değerlendirme Seti (Golden Eval Dataset)
# expected_keywords: Doğru cevapta geçmesi beklenen kritik bilgi parçaları
# in_context: True -> Bilgi teknik notta var | False -> Bilgi notta yok (Halüsinasyon testi)
eval_dataset = [
    {
        "id": "Q01",
        "context": "TB-X İHA azami kalkış ağırlığı (MTOW) 750 kg olup faydalı yük kapasitesi 150 kg'dır.",
        "question": "TB-X İHA'nın azami kalkış ağırlığı ve faydalı yük kapasitesi nedir?",
        "expected_keywords": ["750", "150"],
        "in_context": True
    },
    {
        "id": "Q02",
        "context": "C-Bant görüş hattı (LOS) veri linki menzili açık havada 150 km'dir.",
        "question": "C-Bant veri linki menzili kaç kilometredir?",
        "expected_keywords": ["150"],
        "in_context": True
    },
    {
        "id": "Q03",
        "context": "İniş takımları yerden 150 metre irtifada otonom olarak açılır.",
        "question": "İniş takımları hangi irtifada açılır?",
        "expected_keywords": ["150"],
        "in_context": True
    },
    {
        "id": "Q04",
        "context": "Motor egzoz gazı sıcaklığı (EGT) 780 dereceyi aştığında sistem otomatik güç keser.",
        "question": "Hangi EGT sıcaklığında otomatik güç kesilir?",
        "expected_keywords": ["780"],
        "in_context": True
    },
    {
        "id": "Q05",
        "context": "Yedek LiPo batarya paketi acil durumlarda 25 dakika boyunca aviyonik sistemleri besler.",
        "question": "Yedek LiPo batarya acil durumda kaç dakika dayanır?",
        "expected_keywords": ["25"],
        "in_context": True
    },
    {
        "id": "Q06",
        "context": "INS/GPS hibrit navigasyon modülü saniyede 100 Hz frekansında konum günceller.",
        "question": "Navigasyon modülünün konum güncelleme frekansı kaç Hz'dir?",
        "expected_keywords": ["100"],
        "in_context": True
    },
    {
        "id": "Q07",
        "context": "Otopilot yazılımı RTOS (Gerçek Zamanlı İşletim Sistemi) çekirdeği üzerinde çalışır.",
        "question": "Otopilot yazılımı hangi işletim sistemi çekirdeği üzerinde çalışmaktadır?",
        "expected_keywords": ["rtos"],
        "in_context": True
    },
    {
        "id": "Q08",
        "context": "TB-X İHA azami kalkış ağırlığı (MTOW) 750 kg olup faydalı yük kapasitesi 150 kg'dır.",
        "question": "TB-X İHA'nın kanat açıklığı kaç metredir?",
        "expected_keywords": ["yetersiz bilgi"],
        "in_context": False
    },
    {
        "id": "Q09",
        "context": "C-Bant görüş hattı (LOS) veri linki menzili açık havada 150 km'dir.",
        "question": "Uydu haberleşme (SATCOM) veri aktarım hızı kaç Mbps'tir?",
        "expected_keywords": ["yetersiz bilgi"],
        "in_context": False
    },
    {
        "id": "Q10",
        "context": "Motor egzoz gazı sıcaklığı (EGT) 780 dereceyi aştığında sistem otomatik güç keser.",
        "question": "Motorun yakıt deposu hacmi kaç litredir?",
        "expected_keywords": ["yetersiz bilgi"],
        "in_context": False
    }
]

# A ve B Versiyon Konfigürasyonları (Model + Prompt + Hiperparametre Versioning)
versions = {
    "v1.0-baseline": {
        "system_prompt": "Sen yardımcı bir havacılık asistanısın. Soruları cevapla.",
        "temperature": 0.7,
        "max_new_tokens": 120
    },
    "v2.0-strict-ops": {
        "system_prompt": (
            "Sen bir İHA teknik operasyon asistanısın. SADECE verilen 'Teknik Not' içindeki bilgiyi kullan. "
            "Cevabı tek cümlede, net ve sayısal verilerle ver. "
            "Eğer sorunun cevabı notta açıkça yoksa tahmin yürütme ve sadece 'YETERSİZ BİLGİ' yaz."
        ),
        "temperature": 0.1,
        "max_new_tokens": 60
    }
}
def tr_lower(text: str) -> str:
    return text.replace("İ", "i").replace("I", "ı").lower()
def evaluate_response(answer: str, expected_keywords: list) -> int:
    """Cevabın beklenen tüm anahtar bilgileri içerip içermediğini 1 (Doğru) veya 0 (Hatalı) olarak puanlar."""
    clean_ans = tr_lower(answer)
    return int(all(tr_lower(kw) in clean_ans for kw in expected_keywords))
print(f"Test seti ({len(eval_dataset)} soru) ve versiyonlar ({list(versions.keys())}) tanımlandı.")

Test seti (10 soru) ve versiyonlar (['v1.0-baseline', 'v2.0-strict-ops']) tanımlandı.


In [ ]:
def run_ab_experiment(model, tokenizer, dataset, version_configs):
    experiment_records = []

    for ver_name, cfg in version_configs.items():
        print(f"\n>>> Çalıştırılıyor: {ver_name} (temp={cfg['temperature']}, max_tokens={cfg['max_new_tokens']})")

        for item in dataset:
            user_prompt = f"Teknik Not: {item['context']}\nSoru: {item['question']}"
            messages = [
                {"role": "system", "content": cfg["system_prompt"]},
                {"role": "user", "content": user_prompt}
            ]

            text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = tokenizer(text_input, return_tensors="pt").to(model.device)
            input_tokens = inputs.input_ids.shape[1]

            if torch.cuda.is_available():
                torch.cuda.synchronize()
            t0 = time.time()

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=cfg["max_new_tokens"],
                    do_sample=True,
                    temperature=cfg["temperature"],
                    pad_token_id=tokenizer.eos_token_id
                )

            if torch.cuda.is_available():
                torch.cuda.synchronize()
            latency_sec = time.time() - t0

            generated_ids = outputs[0][input_tokens:]
            output_tokens = len(generated_ids)
            answer_text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

            score = evaluate_response(answer_text, item["expected_keywords"])

            experiment_records.append({
                "timestamp": datetime.now().isoformat(),
                "version": ver_name,
                "question_id": item["id"],
                "in_context": item["in_context"],
                "question": item["question"],
                "answer": answer_text,
                "is_correct": score,
                "latency_sec": round(latency_sec, 3),
                "input_tokens": input_tokens,
                "output_tokens": output_tokens,
                "total_tokens": input_tokens + output_tokens
            })

    log_path = "llmops_experiment_logs.json"
    with open(log_path, "w", encoding="utf-8") as f:
        json.dump(experiment_records, f, ensure_ascii=False, indent=2)

    print(f"\nDeney tamamlandı! Tüm kayıtlar '{log_path}' dosyasına yazıldı.")
    return experiment_records

logs = run_ab_experiment(model, tokenizer, eval_dataset, versions)


>>> Çalıştırılıyor: v1.0-baseline (temp=0.7, max_tokens=120)

>>> Çalıştırılıyor: v2.0-strict-ops (temp=0.1, max_tokens=60)

Deney tamamlandı! Tüm kayıtlar 'llmops_experiment_logs.json' dosyasına yazıldı.


In [ ]:
with open("llmops_experiment_logs.json", "r", encoding="utf-8") as f:
    saved_logs = json.load(f)

df = pd.DataFrame(saved_logs)

print("=" * 85)
print("LLMOps A/B TESTİ ÖZET PERFORMANS RAPORU")
print("=" * 85)

summary = df.groupby("version").agg(
    Toplam_Soru=("question_id", "count"),
    Dogru_Sayisi=("is_correct", "sum"),
    Dogruluk_Yuzde=("is_correct", lambda x: round(x.mean() * 100, 1)),
    Ort_Sure_Sn=("latency_sec", lambda x: round(x.mean(), 3)),
    Ort_Cikis_Token=("output_tokens", lambda x: round(x.mean(), 1)),
    Toplam_Token=("total_tokens", "sum")
).reset_index()

print(summary.to_string(index=False))

print("\n" + "=" * 85)
print("HALÜSİNASYON TUZAĞI SORULARI (Q08 - Q10) KARŞILAŞTIRMASI")
print("=" * 85)

hallucination_df = df[df["in_context"] == False][["version", "question_id", "is_correct", "output_tokens", "answer"]]
for _, row in hallucination_df.iterrows():
    durum = "BAŞARILI (Reddetti)" if row["is_correct"] == 1 else "BAŞARISIZ (Uydurdu)"
    print(f"[{row['version']}] {row['question_id']} | {durum} | Token: {row['output_tokens']}")
    print(f"   Cevap: {row['answer']}\n")

LLMOps A/B TESTİ ÖZET PERFORMANS RAPORU
        version  Toplam_Soru  Dogru_Sayisi  Dogruluk_Yuzde  Ort_Sure_Sn  Ort_Cikis_Token  Toplam_Token
  v1.0-baseline           10             7            70.0        6.398             67.3          1568
v2.0-strict-ops           10            10           100.0        0.288              6.3          1548

HALÜSİNASYON TUZAĞI SORULARI (Q08 - Q10) KARŞILAŞTIRMASI
[v1.0-baseline] Q08 | BAŞARISIZ (Uydurdu) | Token: 120
   Cevap: Üzgünüm ama verdiğiniz bilgilere göre TB-X İHA'nın kanat açıklığını bulamıyorum. Verilen notta MTOW (Maximum Take-Off Weight) ve faydalı yük kapasiteleri hakkında bilgi veriliyor ancak kanat açıklığı veya diğer ana özelliklerin belirtildiği bir şey yok. Bu bilgiler genellikle avuccu kanat açıklığı, vuran hacmi veya başka bir ana özellik hakkında sağlanır. Lütfen bu konuda daha fazla bilgi sağlayınız

[v1.0-baseline] Q09 | BAŞARISIZ (Uydurdu) | Token: 120
   Cevap: Cevap için genellikle detaylı bilgilerinize ihtiyaç duyarım